# Editorial engine — end to end

Load persisted clusters, keep the top 100 by `big_story_score`, then write a newsletter for each reader in `readers.json`.

In [22]:
import importlib
import sys
from pathlib import Path

src = Path(".").resolve() / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

import editorial.config
import editorial.llm
import editorial.s3
import editorial.ranking
import editorial.newsletter
import editorial.main_section
import editorial.headlines
import editorial.stories
import editorial.prompts

for module in (
    editorial.config,
    editorial.prompts,
    editorial.llm,
    editorial.s3,
    editorial.ranking,
    editorial.main_section,
    editorial.headlines,
    editorial.stories,
    editorial.newsletter,
):
    importlib.reload(module)

from editorial.config import EditorialConfig
from editorial.llm import LlmClient
from editorial.s3 import load_persisted, save_newsletters
from editorial.ranking import (
    articles_by_cluster_id,
    mean_article_topic_jaccard,
    top_clusters_by_story_score,
)
from editorial.newsletter import write_for_reader

config = EditorialConfig(job_date=None)
llm = LlmClient(config)
config.resolve_job_date()

'2026-10-01'

In [3]:
config.job_date = "2026-09-30"

## Readers

In [23]:
readers = config.load_readers()
readers

[{'email': 'alex.morgan@example.com',
  'topics': ['politics_and_government', 'business_and_economy', 'technology']},
 {'email': 'sam.smith@example.net', 'topics': ['entertainment_and_culture']}]

## Load persisted clusters, then keep the top 100 stories

In [24]:
loaded = load_persisted(config)
articles = loaded["articles"]
clusters = top_clusters_by_story_score(loaded["clusters"], config.top_n_by_story)

print(loaded["job_date"], len(articles), "articles,", len(loaded["clusters"]), "loaded,", len(clusters), "top stories")
clusters[0] if clusters else None

s3 article-cluster ids: 100%|██████████| 21/21 [00:04<00:00,  4.72it/s]

Loaded date=2026-10-01: 5176 articles, 485 clusters
2026-10-01 5176 articles, 485 loaded, 100 top stories


{'cluster_id': 369,
 'cluster_size': 5,
 'cluster_density': 24.762311547507153,
 'membership_probability': 0.9777838887452454,
 'diversity_reliability': 1.4011398043485328,
 'topic_consistency': 0.8,
 'majority_topic_consistency': 1.0,
 'entity_consistency': 0.2421134421134421,
 'summary': 'GOP crime-attack lines are losing traction; the 2026 midterm campaign is viewed as blundering; doubts about Senate wins in Georgia and North Carolina persist; fundraising (with Trump promises) is crucial to defending the majority; and 10 Democrat-held House seats are most likely to flip.',
 'consistency_score': 2.0421134421134424,
 'big_story_score': 2.8612864287403377,
 'hashed_cluster_id': '889ba3b74083e7e625cb599a8130b256de7c185f907ad49d9cab1ad2e272014b',
 'topics': [{'topic': 'politics_and_government', 'percentage': 1.0},
  {'topic': 'crime_and_legal', 'percentage': 0.2}]}

## Topic match for one cluster

Set `reader_topics` and `cluster_id`, then compute `mean_article_topic_jaccard` over that cluster's articles.

In [38]:
reader_topics = readers[1]["topics"]
cluster_id = 402

members = articles_by_cluster_id(articles).get(cluster_id, [])
score = mean_article_topic_jaccard(reader_topics, members)

{
    "cluster_id": cluster_id,
    "reader_topics": reader_topics,
    "n_articles": len(members),
    "mean_article_topic_jaccard": score,
}

{'cluster_id': 402,
 'reader_topics': ['entertainment_and_culture'],
 'n_articles': 10,
 'mean_article_topic_jaccard': 0.9}

## Write one newsletter per reader

Each loop scores the 100 clusters against that reader's topics (mean article-level Jaccard), keeps the top 20, then runs the three writers.

In [ ]:
newsletters = []
for reader in readers:
    print("\n===", reader["email"], reader["topics"], "===")
    newsletter = write_for_reader(llm, reader, clusters, articles, config)
    newsletters.append(newsletter)
    print(list(newsletter.keys()))
    print(newsletter["main_section"]["title"])
    print([item["headline"] for item in newsletter["headlines"]["headlines"]])
    print([item["headline"] for item in newsletter["stories_to_watch"]["stories_to_watch"]])


=== alex.morgan@example.com ['politics_and_government', 'business_and_economy', 'technology'] ===
Writing newsletter for alex.morgan@example.com (20 clusters)
Main section: The AI Safety Vacuum Is Becoming a Political Problem
Reason: Plan 0 offers the strongest combination of importance, timeliness, coherence, and explanatory value. It unifies major AI developments around the consequential question of governance and accountability, connecting corporate self-regulation, consumer protection, political manipulation, privacy, and presidential power. Its questions support rigorous comparison of voluntary safeguards with enforceable oversight and give the section clear forward-looking value. Plan 1 is important but more diffuse, Plan 2 is substantial but less tightly centered, and Plan 3 is narrower and combines several less connected accountability stories.
Headlines: ['FTC Expands Probes of OpenAI and Anthropic Over AI Consumer Harms', 'Middle East Oil Exports Recover, but Hormuz Tensions

In [14]:
import os
import html
import smtplib

from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText


# ============================================================
# GMAIL CONFIGURATION
# ============================================================


SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587
SMTP_USERNAME = "pricorr6@gmail.com"
SMTP_PASSWORD = "vrsywmpqyojgkhmx"

FROM_EMAIL = "pricorr6@gmail.com"


# ============================================================
# HTML HELPERS
# ============================================================

def escape(text):
    """Safely escape text before putting it into HTML."""
    return html.escape(str(text))


def render_story(story, number=None):
    """
    Render one story.

    Expected format:
    {
        "title": "...",
        "details": "..."
    }
    """

    number_html = ""

    if number is not None:
        number_html = f"""
        <div class="story-number">
            {number}
        </div>
        """

    return f"""
    <tr>
        <td class="story">

            {number_html}

            <div class="story-content">

                <div class="story-title">
                    {escape(story["title"])}
                </div>

                <div class="story-details">
                    {escape(story["details"])}
                </div>

            </div>

        </td>
    </tr>
    """


# ============================================================
# NEWSLETTER HTML
# ============================================================

def generate_newsletter_html(newsletter):
    """
    Generate the complete HTML newsletter.

    Expected structure:

    newsletter = {
        "main": {
            "title": "...",
            "body": "..."
        },

        "headlines": [
            {
                "title": "...",
                "details": "..."
            },
            ...
        ],

        "stories_to_watch": [
            {
                "title": "...",
                "details": "..."
            },
            ...
        ]
    }
    """

    # --------------------------------------------------------
    # Main section
    # --------------------------------------------------------

    main = newsletter["main"]

    main_html = f"""
    <div class="main-title">
        {escape(main["title"])}
    </div>

    <div class="main-body">
        {escape(main["body"])}
    </div>
    """

    # --------------------------------------------------------
    # Headlines
    # --------------------------------------------------------

    headlines = newsletter["headlines"]

    if len(headlines) != 3:
        raise ValueError(
            "The headlines section must contain exactly 3 headlines."
        )

    headlines_html = ""

    for i, headline in enumerate(headlines, start=1):
        headlines_html += render_story(
            headline,
            number=i
        )

    # --------------------------------------------------------
    # Stories to watch
    # --------------------------------------------------------

    stories_to_watch = newsletter["stories_to_watch"]

    if len(stories_to_watch) != 3:
        raise ValueError(
            "The stories_to_watch section must contain exactly 3 stories."
        )

    stories_html = ""

    for i, story in enumerate(stories_to_watch, start=1):
        stories_html += render_story(
            story,
            number=i
        )

    # --------------------------------------------------------
    # Complete HTML
    # --------------------------------------------------------

    return f"""
<!DOCTYPE html>

<html>

<head>

    <meta charset="UTF-8">

    <meta name="viewport"
          content="width=device-width, initial-scale=1.0">

    <!-- Tell email clients that both modes are supported -->
    <meta name="color-scheme"
          content="light dark">

    <meta name="supported-color-schemes"
          content="light dark">

    <style>

        /* ====================================================
           COLOR VARIABLES
           ==================================================== */

        :root {{
            color-scheme: light dark;
        }}


        /* ====================================================
           BASE
           ==================================================== */

        body {{
            margin: 0;
            padding: 0;
            width: 100%;
            background-color: #f4f1ea;
            color: #111111;
            font-family:
                Arial,
                Helvetica,
                sans-serif;
        }}

        table {{
            border-spacing: 0;
        }}

        td {{
            padding: 0;
        }}


        /* ====================================================
           MAIN CONTAINER
           ==================================================== */

        .outer-container {{
            width: 100%;
            background-color: #f4f1ea;
        }}

        .newsletter {{
            width: 620px;
            max-width: 620px;
            background-color: #ffffff;
        }}


        /* ====================================================
           TOP ACCENT
           ==================================================== */

        .accent {{
            height: 7px;
            background-color: #ff5a36;
            font-size: 0;
            line-height: 0;
        }}


        /* ====================================================
           MAIN SECTION
           ==================================================== */

        .main-section {{
            padding: 35px 38px 38px 38px;
        }}

        .main-label {{
            font-size: 12px;
            line-height: 1.2;
            font-weight: 800;
            letter-spacing: 1.5px;
            text-transform: uppercase;
            color: #ff5a36;
            margin-bottom: 10px;
        }}

        .main-title {{
            font-size: 34px;
            line-height: 1.08;
            font-weight: 900;
            color: #111111;
            margin-bottom: 18px;
        }}

        .main-body {{
            font-size: 17px;
            line-height: 1.6;
            color: #444444;
        }}


        /* ====================================================
           SECTION HEADERS
           ==================================================== */

        .section {{
            padding: 0 38px 38px 38px;
        }}

        .section-header {{
            border-top: 3px solid #111111;
            padding-top: 14px;
            margin-bottom: 22px;
        }}

        .section-title {{
            font-size: 22px;
            line-height: 1.2;
            font-weight: 900;
            text-transform: uppercase;
            letter-spacing: 0.5px;
            color: #111111;
        }}


        /* ====================================================
           STORIES
           ==================================================== */

        .story {{
            padding: 0 0 22px 0;
            border-bottom: 1px solid #dddddd;
        }}

        .story-number {{
            float: left;
            width: 30px;
            height: 30px;
            margin-right: 13px;

            background-color: #111111;
            color: #ffffff;

            font-size: 14px;
            line-height: 30px;
            font-weight: 800;
            text-align: center;
        }}

        .story-content {{
            overflow: hidden;
        }}

        .story-title {{
            font-size: 19px;
            line-height: 1.3;
            font-weight: 800;
            color: #111111;
            margin-bottom: 6px;
        }}

        .story-details {{
            font-size: 15px;
            line-height: 1.55;
            color: #555555;
        }}


        /* ====================================================
           SPACING BETWEEN STORIES
           ==================================================== */

        .story-spacer {{
            height: 20px;
        }}


        /* ====================================================
           FOOTER
           ==================================================== */

        .footer {{
            padding: 24px 38px;
            background-color: #111111;
        }}

        .footer-text {{
            font-size: 12px;
            line-height: 1.5;
            color: #aaaaaa;
        }}


        /* ====================================================
           MOBILE
           ==================================================== */

        @media only screen and (max-width: 650px) {{

            .newsletter {{
                width: 100% !important;
                max-width: 100% !important;
            }}

            .main-section {{
                padding: 28px 24px 32px 24px !important;
            }}

            .section {{
                padding-left: 24px !important;
                padding-right: 24px !important;
            }}

            .main-title {{
                font-size: 29px !important;
            }}

            .main-body {{
                font-size: 16px !important;
            }}

            .story-title {{
                font-size: 18px !important;
            }}

        }}


        /* ====================================================
           DARK MODE
           ==================================================== */

        @media (prefers-color-scheme: dark) {{

            body {{
                background-color: #171717 !important;
                color: #eeeeee !important;
            }}

            .outer-container {{
                background-color: #171717 !important;
            }}

            .newsletter {{
                background-color: #242424 !important;
            }}

            .main-title,
            .section-title,
            .story-title {{
                color: #f5f5f5 !important;
            }}

            .main-body,
            .story-details {{
                color: #cccccc !important;
            }}

            .section-header {{
                border-color: #eeeeee !important;
            }}

            .story {{
                border-color: #444444 !important;
            }}

            .story-number {{
                background-color: #eeeeee !important;
                color: #111111 !important;
            }}

        }}

    </style>

</head>


<body>

<table
    class="outer-container"
    width="100%"
    cellpadding="0"
    cellspacing="0"
    border="0"
>

    <tr>

        <td align="center" style="padding: 30px 12px;">

            <table
                class="newsletter"
                width="620"
                cellpadding="0"
                cellspacing="0"
                border="0"
            >

                <!-- =========================================
                     TOP ACCENT
                     ========================================= -->

                <tr>
                    <td class="accent"></td>
                </tr>


                <!-- =========================================
                     MAIN SECTION
                     ========================================= -->

                <tr>

                    <td class="main-section">

                        <div class="main-label">
                            THE BIG PICTURE
                        </div>

                        {main_html}

                    </td>

                </tr>


                <!-- =========================================
                     HEADLINES
                     ========================================= -->

                <tr>

                    <td class="section">

                        <div class="section-header">

                            <div class="section-title">
                                Headlines
                            </div>

                        </div>


                        <table
                            width="100%"
                            cellpadding="0"
                            cellspacing="0"
                            border="0"
                        >

                            {headlines_html}

                        </table>

                    </td>

                </tr>


                <!-- =========================================
                     STORIES TO WATCH
                     ========================================= -->

                <tr>

                    <td class="section">

                        <div class="section-header">

                            <div class="section-title">
                                Stories to Watch
                            </div>

                        </div>


                        <table
                            width="100%"
                            cellpadding="0"
                            cellspacing="0"
                            border="0"
                        >

                            {stories_html}

                        </table>

                    </td>

                </tr>


                <!-- =========================================
                     FOOTER
                     ========================================= -->

                <tr>

                    <td class="footer">

                        <div class="footer-text">
                            Your personalized daily news briefing.
                        </div>

                    </td>

                </tr>

            </table>

        </td>

    </tr>

</table>

</body>

</html>
"""


# ============================================================
# PLAIN TEXT VERSION
# ============================================================

def generate_plain_text(newsletter):
    """
    Generate a plain-text fallback version of the newsletter.
    """

    lines = []

    # Main
    lines.append("THE BIG PICTURE")
    lines.append("")
    lines.append(newsletter["main"]["title"])
    lines.append("")
    lines.append(newsletter["main"]["body"])
    lines.append("")

    # Headlines
    lines.append("====================")
    lines.append("HEADLINES")
    lines.append("====================")
    lines.append("")

    for i, headline in enumerate(
        newsletter["headlines"],
        start=1
    ):
        lines.append(
            f"{i}. {headline['title']}"
        )
        lines.append(
            headline["details"]
        )
        lines.append("")

    # Stories to watch
    lines.append("====================")
    lines.append("STORIES TO WATCH")
    lines.append("====================")
    lines.append("")

    for i, story in enumerate(
        newsletter["stories_to_watch"],
        start=1
    ):
        lines.append(
            f"{i}. {story['title']}"
        )
        lines.append(
            story["details"]
        )
        lines.append("")

    return "\n".join(lines)


# ============================================================
# SEND EMAIL
# ============================================================

def send_newsletter(
    recipient,
    subject,
    newsletter
):
    """
    Generate and send the newsletter through Gmail SMTP.
    """

    html_content = generate_newsletter_html(newsletter)
    plain_text = generate_plain_text(newsletter)

    message = MIMEMultipart("alternative")

    message["Subject"] = subject
    message["From"] = FROM_EMAIL
    message["To"] = recipient

    # Plain-text fallback
    message.attach(
        MIMEText(
            plain_text,
            "plain",
            "utf-8"
        )
    )

    # HTML version
    message.attach(
        MIMEText(
            html_content,
            "html",
            "utf-8"
        )
    )

    # Connect to Gmail
    with smtplib.SMTP(
        SMTP_HOST,
        SMTP_PORT
    ) as server:

        server.starttls()

        server.login(
            SMTP_USERNAME,
            SMTP_PASSWORD
        )

        server.sendmail(
            FROM_EMAIL,
            recipient,
            message.as_string()
        )

    print(
        f"Newsletter successfully sent to {recipient}"
    )




In [ ]:
# ============================================================
# EXAMPLE NEWSLETTER
# ============================================================

if __name__ == "__main__":

    newsletter = {

        # ----------------------------------------------------
        # MAIN
        # ----------------------------------------------------

        "main": {

            "title":
                newsletters[0]['main_section']['title'],

            "body": newsletters[0]['main_section']['body']
        },


        # ----------------------------------------------------
        # HEADLINES
        # ----------------------------------------------------

        "headlines": [

            {
                "title":
                    newsletters[0]['headlines']['headlines'][0]['headline'],

                "details":
                    newsletters[0]['headlines']['headlines'][0]['details']
            },

            {
                "title":
                    newsletters[0]['headlines']['headlines'][1]['headline'],

                "details":
                    newsletters[0]['headlines']['headlines'][1]['details']
            },

            {
                "title":
                    newsletters[0]['headlines']['headlines'][2]['headline'],

                "details":
                    newsletters[0]['headlines']['headlines'][2]['details']
            }

        ],


        # ----------------------------------------------------
        # STORIES TO WATCH
        # ----------------------------------------------------

        "stories_to_watch": [

            {
                "title":
                    newsletters[0]['stories_to_watch']['stories_to_watch'][0]['headline'],

                "details":
                    newsletters[0]['stories_to_watch']['stories_to_watch'][0]['watch_for']
            },

            {
                "title":
                    newsletters[0]['stories_to_watch']['stories_to_watch'][1]['headline'],

                "details":
                    newsletters[0]['stories_to_watch']['stories_to_watch'][1]['watch_for']
            },

            {
                "title":
                    newsletters[0]['stories_to_watch']['stories_to_watch'][2]['headline'],

                "details":
                    newsletters[0]['stories_to_watch']['stories_to_watch'][2]['watch_for']
            }


        ]
    }


    # --------------------------------------------------------
    # SEND
    # --------------------------------------------------------

    send_newsletter(
        recipient="w.elguettabi@gmail.com",
        subject="Your Daily Briefing",
        newsletter=newsletter
    )

Newsletter successfully sent to ahmedmerttab@gmail.com


## Persist newsletters to S3

Each reader dict is written as JSONL under `date={job_date}/`, named by email. A `manifest.json` lists the run.

In [ ]:
manifest = save_newsletters(newsletters, config)
manifest

s3 newsletters:  50%|█████     | 1/2 [00:02<00:02,  2.48s/it]

Wrote s3://readers-newsletters--eun1-az1--x-s3/date=2026-09-29/alex.morgan@example.com.jsonl


s3 newsletters: 100%|██████████| 2/2 [00:03<00:00,  1.55s/it]

Wrote s3://readers-newsletters--eun1-az1--x-s3/date=2026-09-29/sam.smith@example.net.jsonl


Wrote manifest s3://readers-newsletters--eun1-az1--x-s3/date=2026-09-29/manifest.json


{'date': '2026-09-29',
 'status': 'success',
 'started_at': '2026-09-30T05:32:01.703970+00:00',
 'finished_at': '2026-09-30T05:32:04.813848+00:00',
 'bucket': 'readers-newsletters--eun1-az1--x-s3',
 'newsletter_count': 2,
 'newsletters': [{'email': 'alex.morgan@example.com',
   'topics': ['politics_and_government', 'business_and_economy', 'technology'],
   'key': 'date=2026-09-29/alex.morgan@example.com.jsonl',
   'written_at': '2026-09-30T05:32:04.192820+00:00'},
  {'email': 'sam.smith@example.net',
   'topics': ['entertainment_and_culture'],
   'key': 'date=2026-09-29/sam.smith@example.net.jsonl',
   'written_at': '2026-09-30T05:32:04.806570+00:00'}]}

: 

In [36]:
newsletters[1]['headlines']

{'headlines': [{'cluster_id': 338,
   'headline': 'News organizations challenge White House media-access ban in federal court',
   'angle': 'Legal confrontation over presidential press access and media restrictions',
   'details': 'CNN, MS NOW and Politico are seeking a preliminary injunction to keep the administration from reinstating their exclusion while their lawsuit proceeds. They argue the White House has applied the ban inconsistently, including selectively restricting CNN’s pool duties. Major networks plan to continue rotating pool coverage, keeping the dispute consequential for press access and First Amendment protections.',
   'hashed_cluster_id': '9df84b3ee8519b6bbd11747bc8846a86729182ade966c324866f8251f1748101'},
  {'cluster_id': 70,
   'headline': '“Saved by the Bell” actor Dennis Haskins dies at 75',
   'angle': 'Entertainment obituary for the actor best known as Mr. Belding',
   'details': 'Haskins played Principal Richard Belding from the 1989 debut of “Good Morning, Mi